# 10 機器學習 — 參考解答

松柏護理之家退伍軍人症群聚事件機器學習練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

# 特徵定義
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]
feature_cols = num_cols + cat_cols + bin_cols

X = df[feature_cols]
y_infected = df["infected"]
y_severe = df["severe_outcome"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

## 題目 1：class_weight="balanced" 的效果

In [ ]:
# 無 class_weight
clf_default = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
scores_default = cross_val_score(clf_default, X, y_severe, cv=5, scoring="roc_auc")

# 有 class_weight="balanced"
clf_balanced = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42,
                                 class_weight="balanced")),
])
scores_balanced = cross_val_score(clf_balanced, X, y_severe, cv=5, scoring="roc_auc")

print("=== Task B (severe_outcome) ===")
print(f"class_weight=None:       AUC = {scores_default.mean():.3f} \u00b1 {scores_default.std():.3f}")
print(f"class_weight='balanced': AUC = {scores_balanced.mean():.3f} \u00b1 {scores_balanced.std():.3f}")

print("\n\u2192 class_weight='balanced' 會對少數類別給予更高權重")
print("\u2192 在 AUC 上差異通常不大，但對 recall 有幫助")
print("\u2192 當正例比例很低（如 <10%）時，balanced 的效果更明顯")

## 題目 2：Task B 的特徵重要性

In [ ]:
# Random Forest on Task B
clf_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores_rf_b = cross_val_score(clf_rf, X, y_severe, cv=5, scoring="roc_auc")
print(f"Random Forest 5-fold CV AUC (Task B) = {scores_rf_b.mean():.3f} \u00b1 {scores_rf_b.std():.3f}")

# Permutation importance
X_train, X_test, y_train, y_test = train_test_split(
    X, y_severe, test_size=0.3, random_state=42,
)
clf_rf.fit(X_train, y_train)

perm = permutation_importance(
    clf_rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc",
)

imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
        color="#e34a33", alpha=0.8)
ax.set_xlabel("Permutation Importance (AUC decrease)")
ax.set_title("Task B (severe_outcome) \u2014 Feature Importance")
plt.tight_layout()
plt.show()

print("\n=== Top 5 重要特徵（Task B）===")
for _, row in imp_df.nlargest(5, "importance").iterrows():
    print(f"  {row['feature']:25s}  importance = {row['importance']:.4f}")

print("\n\u2192 預測重症的重要特徵可能與預測感染不同")
print("\u2192 暴露因子（shower_use）可能對感染重要，但共病對重症更重要")

## 題目 3（挑戰題）：三模型比較 + ROC 曲線

In [ ]:
# 三個模型
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocess", preprocess),
        ("model", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ]),
}

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_infected, test_size=0.3, random_state=42,
)

# 訓練 + AUC + ROC
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#2c7fb8", "#e34a33", "#41b6c4"]

print("=== Task A Test AUC ===")
for (name, clf), color in zip(models.items(), colors):
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)
    print(f"  {name:25s}  AUC = {auc:.3f}")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves \u2014 Task A (infected)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("\n\u2192 在 280 筆資料上，三個模型的表現通常很接近")
print("\u2192 單次 70/30 split 的結果不穩定，交叉驗證更可靠")
print("\u2192 小樣本的結論要保守解讀，避免過度宣稱模型效能")

### 解讀

- **class_weight**：在不平衡資料中，`balanced` 可提高少數類別的 recall，但 AUC 影響有限
- **Task A vs Task B**：預測感染的關鍵特徵（如 `shower_use`）和預測重症的關鍵特徵（如共病）可能不同，反映不同的因果機制
- **模型選擇**：280 筆資料不足以展現複雜模型的優勢。簡單模型 + 正確的交叉驗證 > 複雜模型 + 不當評估
- **ML vs 迴歸**：ML 強調預測，迴歸強調解釋。兩者互補——如果特徵重要性排序與 adjusted OR 方向一致，結論更具說服力

## 題目 4 解答

In [ ]:
# COVID-19：以人口學與共病預測「重症」（二元分類）
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
rng = np.random.default_rng(1004)
n = 1200
age = rng.integers(20, 90, n)
male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n)
hypertension = rng.integers(0, 2, n)
vaccinated = rng.binomial(1, 0.6, n)
logit = (-6 + 0.06 * age + 0.4 * male + 0.7 * diabetes + 0.5 * hypertension - 1.2 * vaccinated)
severe = rng.binomial(1, 1 / (1 + np.exp(-logit)))
covid = pd.DataFrame({"age": age, "male": male, "diabetes": diabetes,
                      "hypertension": hypertension, "vaccinated": vaccinated, "severe": severe})
print(covid["severe"].value_counts(normalize=True).round(3).to_dict())

X = covid[["age", "male", "diabetes", "hypertension", "vaccinated"]]
y = covid["severe"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
model = Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
model.fit(Xtr, ytr)
proba = model.predict_proba(Xte)[:, 1]
pred = model.predict(Xte)
print(f"Accuracy = {accuracy_score(yte, pred):.3f}")
print(f"ROC-AUC  = {roc_auc_score(yte, proba):.3f}")
print("Confusion matrix:\n", confusion_matrix(yte, pred))
coef = pd.Series(model.named_steps["clf"].coef_[0], index=X.columns).sort_values()
print("\n標準化係數（正=增加重症風險）:")
print(coef.round(3).to_string())
print("解讀：年齡、糖尿病、高血壓提高重症風險；vaccinated 係數為負 → 疫苗具保護作用。")

## 題目 5 解答

In [ ]:
# 登革熱：預測是否進展為重症登革熱 DHF（隨機森林）
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
rng = np.random.default_rng(1005)
n = 1000
age = rng.integers(1, 80, n)
secondary_infection = rng.binomial(1, 0.45, n)   # 二次感染是 ADE 風險
platelet = rng.normal(180, 60, n).clip(20, 400)  # 血小板(千/uL)
days_fever = rng.integers(1, 8, n)
logit = (-2.5 + 1.6 * secondary_infection - 0.012 * platelet + 0.15 * days_fever + 0.01 * age)
dhf = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dengue = pd.DataFrame({"age": age, "secondary_infection": secondary_infection,
                       "platelet": platelet.round(0), "days_fever": days_fever, "dhf": dhf})
print(f"DHF 重症比例：{dengue['dhf'].mean():.1%}")

X = dengue[["age", "secondary_infection", "platelet", "days_fever"]]
y = dengue["dhf"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(Xtr, ytr)
proba = rf.predict_proba(Xte)[:, 1]
print(f"ROC-AUC = {roc_auc_score(yte, proba):.3f}")
print("Confusion matrix:\n", confusion_matrix(yte, rf.predict(Xte)))
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature importances:\n", imp.round(3).to_string())
print(f"解讀：{imp.idxmax()} 最重要——二次感染經 ADE 機制大幅提高重症登革熱風險。")

## 題目 6 解答

In [ ]:
# 結核病：預測治療結果（成功 vs 失敗/中斷），比較兩個模型
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1006)
n = 900
age = rng.integers(18, 85, n)
mdr = rng.binomial(1, 0.15, n)          # 抗藥性
hiv = rng.binomial(1, 0.1, n)
adherence = rng.uniform(0.4, 1.0, n)    # 服藥順從度
logit = (2.0 - 1.8 * mdr - 1.2 * hiv + 3.0 * (adherence - 0.7) - 0.01 * age)
success = rng.binomial(1, 1 / (1 + np.exp(-logit)))
tb = pd.DataFrame({"age": age, "mdr": mdr, "hiv": hiv,
                   "adherence": adherence.round(2), "success": success})
print(f"治療成功率：{tb['success'].mean():.1%}")

X = tb[["age", "mdr", "hiv", "adherence"]]
y = tb["success"]
for name, clf in [("LogReg", LogisticRegression(max_iter=1000)),
                  ("RandomForest", RandomForestClassifier(n_estimators=200, random_state=42))]:
    auc = cross_val_score(clf, X, y, cv=5, scoring="roc_auc").mean()
    print(f"{name}: 5-fold 平均 ROC-AUC = {auc:.3f}")
lr = LogisticRegression(max_iter=1000).fit(X, y)
print("\nadherence 係數 =", round(lr.coef_[0][list(X.columns).index("adherence")], 2),
      "→ 服藥順從度越高，治療成功機率越大（正相關）。")
print("解讀：兩模型 AUC 接近；本例線性關係明確，LogReg 已足夠且可解釋。")

## 題目 7 解答

In [ ]:
# 流感：預測住院（梯度提升樹 + ROC 曲線）
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1007)
n = 1100
age = rng.integers(0, 95, n)
chronic = rng.binomial(1, 0.25, n)
vaccinated = rng.binomial(1, 0.5, n)
onset_to_care = rng.integers(0, 6, n)  # 發病到就醫天數
logit = (-3.5 + 0.05 * age + 1.0 * chronic - 0.8 * vaccinated + 0.25 * onset_to_care)
hosp = rng.binomial(1, 1 / (1 + np.exp(-logit)))
flu = pd.DataFrame({"age": age, "chronic": chronic, "vaccinated": vaccinated,
                    "onset_to_care": onset_to_care, "hospitalized": hosp})
print(f"住院比例：{flu['hospitalized'].mean():.1%}")

X = flu[["age", "chronic", "vaccinated", "onset_to_care"]]
y = flu["hospitalized"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
gb = GradientBoostingClassifier(random_state=42)
gb.fit(Xtr, ytr)
proba = gb.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)
print(f"ROC-AUC = {auc:.3f}")
fpr, tpr, _ = roc_curve(yte, proba)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color="#D97757", label=f"GBM (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="#6B6B6B")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("流感住院預測 ROC 曲線"); ax.legend()
plt.tight_layout(); plt.show()
imp = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importances:\n", imp.round(3).to_string())
print("解讀：vaccinated 提高後住院風險下降（保護作用）；年齡與慢性病為主要風險因子。")

## 題目 8 解答

In [ ]:
# 敗血症：ICU 預後預測 + 特徵重要度（挑戰題）
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1008)
n = 1000
age = rng.integers(18, 95, n)
lactate = rng.normal(2.5, 1.5, n).clip(0.5, 12)   # 乳酸
sofa = rng.integers(0, 18, n)                       # SOFA 分數
wbc = rng.normal(12, 6, n).clip(1, 40)
comorbid = rng.integers(0, 4, n)
logit = (-4 + 0.03 * age + 0.45 * lactate + 0.25 * sofa + 0.3 * comorbid + 0.01 * wbc)
death = rng.binomial(1, 1 / (1 + np.exp(-logit)))
sepsis = pd.DataFrame({"age": age, "lactate": lactate.round(1), "sofa": sofa,
                       "wbc": wbc.round(1), "comorbid": comorbid, "death": death})
print(f"院內死亡比例：{sepsis['death'].mean():.1%}")

X = sepsis[["age", "lactate", "sofa", "wbc", "comorbid"]]
y = sepsis["death"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(Xtr, ytr)
print(f"ROC-AUC = {roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]):.3f}")
perm = permutation_importance(rf, Xte, yte, n_repeats=20, random_state=42, scoring="roc_auc")
pi = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
gini = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nPermutation importance（測試集, AUC 下降量）:\n", pi.round(3).to_string())
print("\nGini(feature_importances_):\n", gini.round(3).to_string())
print(f"\n解讀：{pi.idxmax()}/SOFA 等床邊指標對死亡預測貢獻最大。")
print("permutation_importance 直接量測『打亂該特徵後測試集表現下降多少』，")
print("不像 Gini 會高估高基數(連續)特徵，故較能反映真實預測貢獻。")